# Pretrain the ball tracker on public TrackNet data

Pretrains the heatmap tracker on a public small-fast-ball dataset (tennis/badminton TrackNet sets) so it learns to follow a tiny fast ball; you then fine-tune on your own cricket footage.

**Research / non-commercial use only** — these datasets are licensed for research. A model trained on them must not ship in a commercial product; the shipped model is trained on your own data.

**You provide the dataset** — this notebook does not download it (the TrackNet sets live on their own project links). Follow the steps below.

In [ ]:
!git clone --depth 1 https://github.com/LK-maker-007/Snarl.git
%cd Snarl/ml
!pip install -q -e '.[convert]'

## 1. Provide the dataset

Download a TrackNet tennis/badminton dataset from its project repo and extract it so that **each clip is an immediate subdirectory** of `DATA_ROOT`, containing that clip's frame images plus its TrackNet label CSV:

```
DATA_ROOT/
  clip0/  0.png 1.png ...  Label.csv
  clip1/  ...               Label.csv
```

Upload/extract into `/content/data`, or mount Google Drive. Then set `DATA_ROOT` and the frame extension below.

In [ ]:
DATA_ROOT = '/content/data'   # point this at your extracted TrackNet dataset
IMAGE_GLOB = '*.png'          # frame extension in your dataset (e.g. *.png or *.jpg)

# Put your dataset under DATA_ROOT first (download + extract, or mount Drive).
# This notebook does not download it -- the TrackNet sets live on their own project links.
import os
if os.path.isdir(DATA_ROOT):
    print('clips found:', sorted(os.listdir(DATA_ROOT)))
else:
    print(DATA_ROOT, 'not found -- add your dataset, then re-run this cell')

## 2. Normalize labels to our schema

Converts each clip's TrackNet `Frame/Visibility/X/Y` CSV into the `frame,visibility,x,y` `labels.csv` the loader reads.

In [ ]:
!python -m snarl_ml.prepare '{DATA_ROOT}'

## 3. Pretrain

Trains the tracker on the dataset, reporting per-epoch loss and decoded pixel error, and saves the best checkpoint to `/content/pretrained.pt`.

In [ ]:
!python -m snarl_ml.train --image-dir '{DATA_ROOT}' --image-glob '{IMAGE_GLOB}' --epochs 30 --out /content/pretrained.pt

## Next

`/content/pretrained.pt` is a research pretrained tracker — **download it before the session ends.** Fine-tune it on your own consented cricket clips (see `ml/DATA_COLLECTION.md`):

```
python -m snarl_ml.train --image-dir <cricket-clips> --init-from pretrained.pt --epochs 50
```